In [ ]:
# input
aligned_fasta = "./tmp/266_cluster_seqs_pred_aligned.fasta"
high_conf_pred = "../../../../../predict_afdb/data/pred_ge_3_clique_3.tsv"
# output
aligned_result_file = "./tmp/409_aligned_result.tsv"

In [2]:
import pandas as pd
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord

alignments: list[SeqRecord] = []
for i in SeqIO.parse(aligned_fasta, "fasta"):
    alignments.append(i)

ids = set([i.id for i in alignments])
df_pred = pd.read_table(high_conf_pred, usecols=["seq_id", "posi"])
df_pred['seq_id'] = df_pred['seq_id'].map(lambda x: x.split("-")[1])
df_pred = df_pred[df_pred['seq_id'].map(lambda x: x in ids)]

In [4]:
id_to_pred = dict()
for _, row in df_pred.iterrows():
    id_to_pred[row["seq_id"]] = set(
        [int(i) for i in row["posi"].split(",")]
    )

#### get aligned result, copied from "../../predict_afdb/analysis/seq_cluster/scripts/align_pred.py"

In [5]:
def real_posi_to_aligned_posi(seq: str):
    result = dict()
    real_posi = -1
    for idx, aa in enumerate(seq):
        if aa != "-":
            real_posi += 1
            result[real_posi] = idx

    return result

id_to_align_result = []

# first collect all pred aligned posis
all_pred_aligned_posis = set()
for r in alignments:
    r: SeqRecord
    if r.id in id_to_pred:
        seq = str(r.seq)
        real_to_aligned = real_posi_to_aligned_posi(seq)
        real_pred_posis = id_to_pred[r.id]
        for p in real_pred_posis:
            aligned_posi = real_to_aligned[p]
            all_pred_aligned_posis.add(aligned_posi)

# then check each seq in the aligned positions
all_pred_aligned_posis = sorted(list(all_pred_aligned_posis))
for r in alignments:
    seq = str(r.seq)
    real_to_aligned = real_posi_to_aligned_posi(seq)
    aligned_to_real = dict(
        zip(real_to_aligned.values(), real_to_aligned.keys())
    )

    real_pred_posis = id_to_pred[r.id] if r.id in id_to_pred else set()
    real_posis = []
    real_aas = []
    aligned_posis = []
    is_preds = []

    for p in all_pred_aligned_posis:
        real_posi = -1 if p not in aligned_to_real else aligned_to_real[p]
        real_aa = "-" if p not in aligned_to_real else seq[p]
        aligned_posi = -1 if p not in aligned_to_real else p
        is_pred = 0 if real_posi not in real_pred_posis else 1
        real_posis.append(real_posi)
        real_aas.append(real_aa)
        aligned_posis.append(aligned_posi)
        is_preds.append(is_pred)

    id_to_align_result.append(
        {
            "seq_id": r.id,
            "real_posi": ",".join([str(i) for i in real_posis]),
            "real_aa": ",".join(real_aas),
            "aligned_posi": ",".join([str(i) for i in aligned_posis]),
            "is_pred": ",".join([str(i) for i in is_preds]),
        }
    )      

In [6]:
from collections import Counter

def get_site_id(pred_str, sp: int):
    is_pred = pred_str.split(",")
    site_1 = is_pred[:sp]
    site_2 = is_pred[sp:]
    has_pred_in_site_1 = any([i == "1" for i in site_1])
    has_pred_in_site_2 = any([i == "1" for i in site_2])
    if has_pred_in_site_1 and has_pred_in_site_2:
        site_id = "1,2"
    elif has_pred_in_site_1 and not has_pred_in_site_2:
        site_id = "1"
    elif not has_pred_in_site_1 and has_pred_in_site_2:
        site_id = "2"
    else:
        site_id = "0"
    return site_id

df_align = pd.DataFrame(id_to_align_result)
df_pred = df_align[df_align['is_pred'].map(lambda x: x.count("1") > 0)].copy().reset_index(drop=True)

split_posi_to_site = dict()
split_range = range(1, df_pred.iloc[0].is_pred.count(","))

for sp in split_range:
    site = []
    for _, row in df_pred.iterrows():
        site_id = get_site_id(row['is_pred'], sp)
        site.append(site_id)
    split_posi_to_site[sp] = Counter(site)  

In [ ]:
split_posi_to_site
split_posi = 12
# split posi 12

{1: Counter({'2': 407, '1,2': 2}),
 2: Counter({'2': 407, '1,2': 2}),
 3: Counter({'2': 407, '1,2': 2}),
 4: Counter({'2': 407, '1,2': 1, '1': 1}),
 5: Counter({'2': 407, '1,2': 1, '1': 1}),
 6: Counter({'2': 407, '1': 2}),
 7: Counter({'1,2': 285, '2': 122, '1': 2}),
 8: Counter({'1,2': 287, '2': 120, '1': 2}),
 9: Counter({'1,2': 287, '2': 120, '1': 2}),
 10: Counter({'1,2': 287, '2': 120, '1': 2}),
 11: Counter({'1': 200, '2': 120, '1,2': 89}),
 12: Counter({'1': 289, '2': 120}),
 13: Counter({'1': 289, '1,2': 120}),
 14: Counter({'1': 289, '1,2': 120}),
 15: Counter({'1': 289, '1,2': 120}),
 16: Counter({'1': 289, '1,2': 120}),
 17: Counter({'1': 289, '1,2': 120}),
 18: Counter({'1': 322, '1,2': 87})}

In [ ]:
df_align['site_id'] = df_align['is_pred'].map(lambda x: get_site_id(x, split_posi))
df_align.to_csv(aligned_result_file, sep="\t", index=None)